# AfterMath — Data Prep

Build the before/after crop dataset and manifest from raw xBD images.

In [1]:
import sys
sys.path.insert(0, '..')  # so `utils` (repo root) is importable when cwd is notebooks/

import glob
import os
import yaml
import pandas as pd
from PIL import Image
import numpy as np

from utils.xbd_labels import parse_label_file
from utils.cropping import crop_and_resize
from utils.prepare import split_storms, manifest_row

config = yaml.safe_load(open('../config.yaml'))
raw_dir = '../' + config['data']['raw_dir']
processed_dir = '../' + config['data']['processed_dir']
crop_size = config['data']['crop_size']
train_storms, test_storms = split_storms(config['data']['storms'], config['data']['test_storm'])

os.makedirs(f'{processed_dir}/pre', exist_ok=True)
os.makedirs(f'{processed_dir}/post', exist_ok=True)

## Build crops and manifest

Note on the real xBD label schema: post-disaster label files annotate each
building with a `subtype` damage class; pre-disaster label files only contain
building footprints (no `subtype` at all, since damage is undefined before the
disaster). `parse_label_file` returns pre-disaster buildings with
`damage_class=None` so they can still be matched to their post-disaster
counterpart by `uid` and cropped — the manifest always uses the **post**-disaster
label's damage class.

In [2]:
def storm_split(storm):
    return 'train' if storm in train_storms else 'test'

rows = []
crop_id = 0
for storm in config['data']['storms']:
    split = storm_split(storm)
    # normalize to forward slashes: on Windows, glob() joins the matched
    # filename onto the directory prefix with a backslash, which breaks the
    # '/labels/' -> '/images/' string substitution below
    post_label_files = [
        p.replace(os.sep, '/')
        for p in glob.glob(f'{raw_dir}/{storm}/labels/*_post_disaster.json')
    ]
    for label_file in post_label_files:
        pre_label_file = label_file.replace('_post_disaster', '_pre_disaster')
        post_image_file = label_file.replace('/labels/', '/images/').replace('.json', '.png')
        pre_image_file = pre_label_file.replace('/labels/', '/images/').replace('.json', '.png')

        post_labels = {l.uid: l for l in parse_label_file(label_file)}
        pre_labels = {l.uid: l for l in parse_label_file(pre_label_file)}

        post_image = np.array(Image.open(post_image_file).convert('RGB'))
        pre_image = np.array(Image.open(pre_image_file).convert('RGB'))

        for uid, post_label in post_labels.items():
            if uid not in pre_labels:
                continue
            pre_label = pre_labels[uid]

            post_crop = crop_and_resize(post_image, post_label.polygon, size=crop_size)
            pre_crop = crop_and_resize(pre_image, pre_label.polygon, size=crop_size)

            pre_path = f'{processed_dir}/pre/{crop_id}.png'
            post_path = f'{processed_dir}/post/{crop_id}.png'
            Image.fromarray(pre_crop).save(pre_path)
            Image.fromarray(post_crop).save(post_path)

            assert post_label.damage_class is not None, (
                f"post-disaster building {uid} in {storm} has no damage_class "
                f"(subtype missing from label JSON) -- this should never happen for "
                f"post-disaster labels; check for data corruption"
            )
            rows.append(manifest_row(pre_path, post_path, post_label.damage_class, storm, split))
            crop_id += 1

manifest = pd.DataFrame(rows)
manifest.to_csv(f'{processed_dir}/manifest.csv', index=False)
manifest['damage_class'].value_counts()

damage_class
no-damage       33215
minor-damage    14550
major-damage    13330
destroyed        3359
Name: count, dtype: int64

## Sanity checks

In [3]:
print('Total crops:', len(manifest))
print()
print(manifest.groupby('storm')['split'].first())
print()
print(manifest.groupby(['storm', 'damage_class']).size())

Total crops: 64454

storm
hurricane-florence    train
hurricane-harvey      train
hurricane-matthew     train
hurricane-michael      test
Name: split, dtype: object

storm               damage_class
hurricane-florence  destroyed          54
                    major-damage     1245
                    minor-damage      132
                    no-damage        4689
hurricane-harvey    destroyed         401
                    major-damage     8238
                    minor-damage     2663
                    no-damage       11423
hurricane-matthew   destroyed        2147
                    major-damage     1945
                    minor-damage     6548
                    no-damage        2515
hurricane-michael   destroyed         757
                    major-damage     1902
                    minor-damage     5207
                    no-damage       14588
dtype: int64
